In [0]:
%pip install databricks-sdk --upgrade

In [0]:
import re
CATALOG = dbutils.widgets.get("CATALOG")
REFUNDMANAGER_LAKEBASE_INSTANCE_NAME = re.sub(r'[^a-z0-9-]', '-', f"{CATALOG}refundmanager".lower())

In [ ]:
import sys
sys.path.append('../utils')
from uc_state import add

In [0]:
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

WAREHOUSE_NAME = f"{CATALOG}-warehouse"
existing_wh = [wh for wh in w.warehouses.list() if wh.name == WAREHOUSE_NAME]
if existing_wh:
    warehouse = existing_wh[0]
    print(f"♻️ Using existing warehouse: {warehouse.id}")
else:
    warehouse = w.warehouses.create(
        name=WAREHOUSE_NAME,
        cluster_size="2X-Small",
        max_num_clusters=1,
        min_num_clusters=1,
        enable_serverless_compute=True,
    ).result()
    add(CATALOG, "warehouses", warehouse)
    print(f"✅ Created warehouse: {warehouse.id}")

In [0]:
print(warehouse)

In [0]:
app_yaml_contents = f"""command:
  - uvicorn 
  - app.main:app
env:
  - name: DATABRICKS_WAREHOUSE_ID
    value: '{warehouse.id}'
  - name: DATABRICKS_CATALOG
    value: '{CATALOG}'
"""

# for some reason we need to remove it first before this works
import os
if os.path.exists("../apps/refund-manager/app.yaml"):
    os.remove("../apps/refund-manager/app.yaml")

import time
time.sleep(3)

with open("../apps/refund-manager/app.yaml", "w") as f:
    f.write(app_yaml_contents)

In [0]:
from databricks.sdk.service.apps import App, AppResource, AppResourceSqlWarehouse, AppResourceDatabase, AppResourceSqlWarehouseSqlWarehousePermission, AppResourceDatabaseDatabasePermission
import os
import re as _re

source_code_path = os.path.abspath("../apps/refund-manager")

# P1-18: catalog-scope the app name so two users on the same workspace don't
# fight over the global `refundmanager` slug.  Sanitise to lowercase
# alphanumerics+hyphens (Databricks Apps constraint) and cap at 30 chars.
APP_NAME = _re.sub(r"-+", "-", _re.sub(r"[^a-z0-9-]", "-", f"refundmanager-{CATALOG}".lower())).strip("-")[:30]
print(f"App name: {APP_NAME}")

app_def = App(
    name=APP_NAME,
    default_source_code_path=source_code_path,
    resources=[
        AppResource(
            name="sql-warehouse",
            sql_warehouse=AppResourceSqlWarehouse(
                id=warehouse.id,
                permission=AppResourceSqlWarehouseSqlWarehousePermission.CAN_USE),
        ),
        AppResource(
            name="database",
            database=AppResourceDatabase(
                instance_name=f"{REFUNDMANAGER_LAKEBASE_INSTANCE_NAME}",
                database_name="caspers",
                permission=AppResourceDatabaseDatabasePermission.CAN_CONNECT_AND_CREATE,
            ),
        ),
    ],
)

try:
    existing_app = w.apps.get(APP_NAME)
    print(f"♻️ App {APP_NAME} already exists, updating...")
    app = w.apps.update(APP_NAME, app_def)
except Exception:
    app = w.apps.create(app_def)

In [0]:
import time

def _app_state(a):
    cs = getattr(a, "compute_status", None)
    s = getattr(cs, "state", None) if cs is not None else None
    if s is None:
        s = getattr(a, "state", None)
    return getattr(s, "value", str(s)) if s is not None else ""

deadline = time.time() + 30 * 60
while True:
    current = w.apps.get(APP_NAME)
    state = _app_state(current)
    print(f"App {APP_NAME} state: {state}")
    if state in ("ACTIVE", "RUNNING", "READY"):
        app_status = current
        break
    if state in ("ERROR", "FAILED"):
        raise RuntimeError(f"App {APP_NAME} entered failure state: {state}")
    if time.time() > deadline:
        raise TimeoutError(f"App {APP_NAME} not ready after 30 minutes (last state: {state})")
    time.sleep(15)

add(CATALOG, "apps", app_status)

In [ ]:
# grant app permissions to catalog, schema, `all_events` table for reading events
from databricks.sdk.service import catalog

w.grants.update(
    full_name=f"{CATALOG}",
    securable_type="CATALOG",
    changes=[
        catalog.PermissionsChange(
            add=[catalog.Privilege.USE_CATALOG],
            principal=app_status.id
        )
    ]
)

w.grants.update(
    full_name=f"{CATALOG}.lakeflow",
    securable_type="SCHEMA",
    changes=[
        catalog.PermissionsChange(
            add=[catalog.Privilege.USE_SCHEMA],
            principal=app_status.id
        )
    ]
)

w.grants.update(
    full_name=f"{CATALOG}.lakeflow.all_events",
    securable_type="TABLE",
    changes=[
        catalog.PermissionsChange(
            add=[catalog.Privilege.SELECT],
            principal=app_status.id
        )
    ]
)

In [0]:
from databricks.sdk.common.types.fieldmask import FieldMask

from databricks.sdk.service.postgres import (
      Role,
      RoleRoleSpec,
      RoleMembershipRole,
      RoleAuthMethod,
      RoleIdentityType,
      RoleAttributes,
)

PROJECT = f"projects/{REFUNDMANAGER_LAKEBASE_INSTANCE_NAME}"

# P1-17: pick the production branch by `status.default`, not by list index [0].
# The Postgres API does not guarantee list ordering and a future schema bump
# (e.g. adding a `staging` branch by default) would silently elect the wrong
# branch and the GRANT would land on the wrong Postgres database.
branches = list(w.postgres.list_branches(PROJECT))
PRODUCTION = next(
    (b for b in branches if getattr(getattr(b, "status", None), "default", False)),
    branches[0],
).name

# P1-17: pick the app's role by service-principal identity, not by list index
# [1].  The platform auto-creates a role for the app SP when the
# AppResourceDatabase is bound, but its position in the list is not stable —
# we have seen `[0]` be the human admin role on some workspaces and `[1]` on
# others, so the old hack worked by accident.
app_sp_id = (
    getattr(app_status, "service_principal_client_id", None)
    or (app_status.as_dict() if hasattr(app_status, "as_dict") else {}).get("service_principal_client_id")
    or (app_status.as_dict() if hasattr(app_status, "as_dict") else {}).get("id")
)
assert app_sp_id, "Could not determine refund-manager app service principal ID"

# Step 1: try matching on the list-API response.  `list_roles` often returns
# a stripped projection where `spec.postgres_role` is None, so this is a fast
# path for the case where it isn't.
all_roles = list(w.postgres.list_roles(PRODUCTION))
APP_ROLE = next(
    (r for r in all_roles if getattr(getattr(r, "spec", None), "postgres_role", None) == app_sp_id),
    None,
)

# Step 2: list_roles returned None for postgres_role on every role — call
# get_role to fetch the full spec for each and try matching again.  This is
# the path we hit in production: `list_roles` strips spec.postgres_role but
# `get_role` returns the full Role.
if APP_ROLE is None:
    detailed_roles = []
    for r in all_roles:
        try:
            detailed_roles.append(w.postgres.get_role(name=r.name))
        except Exception as e:
            print(f"⚠️  Could not get_role({r.name}): {e}")
    APP_ROLE = next(
        (r for r in detailed_roles if getattr(getattr(r, "spec", None), "postgres_role", None) == app_sp_id),
        None,
    )

# Step 3: still nothing — the platform didn't expose a matching `postgres_role`
# on any role visible to us.  Try to create a SUPERUSER role explicitly for
# the SP (same pattern operational_app.ipynb uses).  If the platform tells us
# a role with that name already exists, fall through to Step 4 (heuristic
# match on identity_type=SERVICE_PRINCIPAL).
if APP_ROLE is None:
    print(
        f"⚠️  No role matched app SP {app_sp_id} in {PRODUCTION}.  "
        f"Attempting to create one explicitly with DATABRICKS_SUPERUSER membership."
    )
    try:
        new_role = w.postgres.create_role(
            parent=PRODUCTION,
            role=Role(
                spec=RoleRoleSpec(
                    identity_type=RoleIdentityType.SERVICE_PRINCIPAL,
                    postgres_role=app_sp_id,
                    membership_roles=[RoleMembershipRole.DATABRICKS_SUPERUSER],
                ),
            ),
        )
        print(f"✅ Created SUPERUSER role {getattr(new_role, 'name', '<unknown>')} for app SP {app_sp_id}")
        APP_ROLE = "created"  # sentinel so we skip the update path below
    except Exception as create_err:
        if "already exists" not in str(create_err).lower():
            raise
        print(
            f"ℹ️  create_role rejected with 'already exists' — role for this SP is "
            f"already provisioned but our lookups didn't surface it.  Falling "
            f"through to identity-type heuristic."
        )
        # Step 4: heuristic — the SP must own *some* role in this branch, but
        # neither list_roles nor get_role exposed the postgres_role field in
        # a way we could match.  Re-fetch with full detail and dump for the
        # logs so we can debug later, then pick the *first* role whose
        # identity_type is SERVICE_PRINCIPAL (there is typically exactly one
        # SP role per app — the deploying human is identity_type=USER).
        sp_roles = []
        for r in all_roles:
            try:
                detail = w.postgres.get_role(name=r.name)
                spec = getattr(detail, "spec", None)
                idt = getattr(spec, "identity_type", None)
                pr = getattr(spec, "postgres_role", None)
                mr = getattr(spec, "membership_roles", None)
                print(f"  • {r.name}: identity_type={idt}, postgres_role={pr!r}, membership_roles={mr}")
                if idt == RoleIdentityType.SERVICE_PRINCIPAL:
                    sp_roles.append(detail)
            except Exception as e:
                print(f"  • {r.name}: get_role failed ({e})")

        if len(sp_roles) == 1:
            APP_ROLE = sp_roles[0]
            print(f"✅ Selected sole SERVICE_PRINCIPAL role {APP_ROLE.name} for promotion.")
        elif len(sp_roles) > 1:
            # Multiple SP roles exist — pick the one whose postgres_role contains
            # the app_sp_id (substring match handles different prefixings) or
            # the most recently created one as a last resort.
            candidate = next(
                (r for r in sp_roles
                 if app_sp_id in str(getattr(getattr(r, "spec", None), "postgres_role", ""))),
                None,
            )
            if candidate is None:
                candidate = sp_roles[-1]
                print(
                    f"⚠️  Multiple SERVICE_PRINCIPAL roles exist and none matched "
                    f"{app_sp_id} by substring.  Picking last-listed: {candidate.name}."
                )
            APP_ROLE = candidate
            print(f"✅ Selected SERVICE_PRINCIPAL role {APP_ROLE.name} for promotion.")
        else:
            # Nothing — the role exists per the platform but we can't see it.
            # Don't fail the task; the app will fall back to its default
            # connect-as-itself behaviour and a human can fix this later if
            # the app's Postgres queries 401.
            print(
                f"⚠️  No SERVICE_PRINCIPAL role visible to us, but create_role "
                f"said one already exists.  Skipping promotion — the app may "
                f"need a manual GRANT pass if it can't connect.  This is not "
                f"fatal for the bundle run."
            )
            APP_ROLE = None

if APP_ROLE not in (None, "created"):
    APP_ROLE.spec = RoleRoleSpec(
        membership_roles=[RoleMembershipRole.DATABRICKS_SUPERUSER]
    )
    w.postgres.update_role(
        name=APP_ROLE.name,
        role=APP_ROLE,
        update_mask=FieldMask(["spec.membership_roles"]),
    )
    print(f"✅ Promoted role {APP_ROLE.name} (app SP {app_sp_id}) to DATABRICKS_SUPERUSER")

In [0]:
app_status

In [0]:
import time
from databricks.sdk.service.apps import AppDeployment

deployment = w.apps.deploy(
    app_name=app_status.name,
    app_deployment=AppDeployment(
        source_code_path=source_code_path
    )
)

def _deploy_state(d):
    st = getattr(d, "status", None)
    s = getattr(st, "state", None) if st is not None else None
    return getattr(s, "value", str(s)) if s is not None else ""

deadline = time.time() + 30 * 60
while True:
    current_dep = w.apps.get_deployment(app_name=app_status.name, deployment_id=deployment.deployment_id)
    state = _deploy_state(current_dep)
    print(f"Deployment state: {state}")
    if state == "SUCCEEDED":
        deployment_status = current_dep
        break
    if state in ("FAILED", "STOPPED"):
        raise RuntimeError(f"Deployment failed for {app_status.name}: state={state}")
    if time.time() > deadline:
        raise TimeoutError(f"Deployment for {app_status.name} not ready after 30 minutes (last state: {state})")
    time.sleep(10)

display(deployment_status)